In [1]:
import pandas as pd

df = pd.read_csv("../data/winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [2]:
df.quality.value_counts().sort_index()

quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

In [3]:
y = df.pop("quality")
X = df

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
)
X_train.shape, X_test.shape

((1199, 11), (400, 11))

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svr", SVR()),
    ]
)

param_grid = {
    "svr__C": [0.1, 1, 10],
    "svr__gamma": ["scale", 0.01, 0.1, 1],
    "svr__epsilon": [0.1, 0.3, 0.5],
}

estimator = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
estimator.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid={'svr__C': [0.1, 1, 10],
                         'svr__epsilon': [0.1, 0.3, 0.5],
                         'svr__gamma': ['scale', 0.01, 0.1, 1]},
             scoring='neg_mean_squared_error')

In [5]:
estimator.best_params_, -estimator.best_score_

({'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma': 'scale'},
 0.41313301377878975)

In [6]:
results = pd.DataFrame(estimator.cv_results_)
results[["params", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score").head(10)

,params,mean_test_score,rank_test_score
12,"{'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma...",-0.413133,1
14,"{'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma...",-0.413251,2
16,"{'svr__C': 1, 'svr__epsilon': 0.3, 'svr__gamma...",-0.414336,3
18,"{'svr__C': 1, 'svr__epsilon': 0.3, 'svr__gamma...",-0.414924,4
17,"{'svr__C': 1, 'svr__epsilon': 0.3, 'svr__gamma...",-0.426015,5
13,"{'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma...",-0.426610,6
29,"{'svr__C': 10, 'svr__epsilon': 0.3, 'svr__gamm...",-0.435483,7
25,"{'svr__C': 10, 'svr__epsilon': 0.1, 'svr__gamm...",-0.435818,8
0,"{'svr__C': 0.1, 'svr__epsilon': 0.1, 'svr__gam...",-0.437508,9
20,"{'svr__C': 1, 'svr__epsilon': 0.5, 'svr__gamma...",-0.437577,10


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred_test = estimator.predict(X_test)
mean_squared_error(y_test, y_pred_test), mean_absolute_error(y_test, y_pred_test)

(0.3490916041179509, 0.45237959059066923)

In [8]:
import os
import pickle

os.makedirs("../submission", exist_ok=True)

with open("../submission/estimator.pkl", "wb") as file:
    pickle.dump(estimator, file)